# Missão Aurora Siger: Verificação de segurança pré-decolagem
---
**Autores:** André Raposo (RM568837), Google Gemini  

## 1. Geração de dados regulares
Inicia a geração de dados de telemetria utilizando as bibliotecas `pandas` e `numpy`, considerando um cenário ideal onde todos os dados estão dentro da faixa de limites esperada.
 

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

n_rows = 500
start_time = datetime.now()

# Gera base de dados com valores randomicos dentro do limite.
data = {
    "timestamp": [start_time + timedelta(minutes=i) for i in range(n_rows)],
    "internal_temp_c": np.random.uniform(18, 35, n_rows),
    "external_temp_c": np.random.uniform(-5, 30, n_rows),
    "battery_voltage_v": np.random.uniform(46, 52, n_rows),
    "battery_current_a": np.random.uniform(20, 120, n_rows),
    "battery_soc_percentage": np.random.uniform(60, 100, n_rows),
    "battery_capacity_ah": np.random.uniform(80, 120, n_rows),
    "power_load_kw": np.random.uniform(5, 25, n_rows),
    "energy_loss_percent": np.random.uniform(2, 8, n_rows),
    "estimated_takeoff_consumption_kw": np.random.uniform(45, 180, n_rows),
    "tank_pressure_bar": np.random.uniform(95, 145, n_rows),
    "structural_integrity": 1,
    "critical_modules_status": 1,
    "telemetry_link_status": 1,
    "is_anomaly": False
}

# Carrega os dados em um dataframe
df = pd.DataFrame(data)

## 2. Geração de dados com anomalias
Introduz um grau de 5% de anomalias nos dados gerados anteriormente

---
O trecho de código foi parcialmente criado utilizando o Google Gemini com o seguinte prompt:   

> "Com base nos dados gerados anteriormente introduza anomalia nos dados de telemetria
>
> Regras: 
> - Insira anomalia em 5% dos dados 
> - Temperatura interna entre 45 e 70 
> - Nível de bateria entre 5 e 25 
> - Pressão fora da faixa (40-80 ou 160-220) 
> - Integridade estrutural 0 
> - Estado dos módulos críticos 0 
> - Estados link de telemetria 0 
>
> Saída 
> - Arquivo CSV válido"

O código foi avaliado e refatorado para remover "alucionações" e melhorar a legibilidade:   


**Alterações:**
- Um dos tipos de anomalia estava com um erro no nome
- Utiliza enum para definir tipos de anomalia `AnomalyTypes` para evitar possíveis erros de digitação.

In [2]:
# Define a porcentagem de anomalia nos dados: 5%
from enum import Enum
from typing import List


n_anomalies = int(len(df) * 0.05)

# Busca por indices aleatóriamente para introduzir anomalias
anomaly_idx = np.random.choice(df.index, n_anomalies, replace=False)


# Define os tipos de dados que poderão ter anomalias
# Define Anomaly Types using an Enum
class AnomalyTypes(Enum):
    INTERNAL_TEMP = 'internal_temp_c'
    BATTERY_SOC = 'battery_soc_percentage'
    TANK_PRESSURE = 'tank_pressure_bar'
    STRUCTURAL_INTEGRITY = 'structural_integrity'
    CRITICAL_MODULES_STATUS = 'critical_modules_status'
    TELEMETRY_LINK_STATUS = 'telemetry_link_status'

anomaly_types: List = list(AnomalyTypes)

for idx in anomaly_idx:
    df.loc[idx, "is_anomaly"] = True
    anomaly_type = np.random.choice(anomaly_types)

    match anomaly_type:
        case AnomalyTypes.INTERNAL_TEMP:
            df.loc[idx, anomaly_type.value] = np.random.uniform(45, 70)
        case AnomalyTypes.BATTERY_SOC:
            df.loc[idx, anomaly_type.value] = np.random.uniform(5, 25)
        case AnomalyTypes.TANK_PRESSURE:
            if np.random.rand() > 0.5:
                df.loc[idx, anomaly_type.value] = np.random.uniform(40, 80)
            else:
                df.loc[idx, anomaly_type.value] = np.random.uniform(160, 220)
        case AnomalyTypes.STRUCTURAL_INTEGRITY:
            df.loc[idx, anomaly_type.value] = 0
        case AnomalyTypes.CRITICAL_MODULES_STATUS:
            df.loc[idx, anomaly_type.value] = 0
        case AnomalyTypes.TELEMETRY_LINK_STATUS:
            df.loc[idx, anomaly_type.value] = 0


df.to_csv("telemetria_simulada.csv", index=False)
df.head()

,timestamp,internal_temp_c,external_temp_c,battery_voltage_v,battery_current_a,battery_soc_percentage,battery_capacity_ah,power_load_kw,energy_loss_percent,estimated_takeoff_consumption_kw,tank_pressure_bar,structural_integrity,critical_modules_status,telemetry_link_status,is_anomaly
0,2026-03-25 22:07:42.147255,34.473606,25.185036,46.386056,51.036292,81.440589,119.163745,7.285487,2.854403,136.894277,131.103790,1,1,1,False
1,2026-03-25 22:08:42.147255,25.177850,-4.188213,48.465421,114.629270,75.429676,103.886668,8.477682,2.392274,125.862344,135.912748,1,1,1,False
2,2026-03-25 22:09:42.147255,22.049348,25.942289,48.137919,29.517034,85.197610,114.722101,12.671776,2.339611,73.772484,112.410687,1,1,1,False
3,2026-03-25 22:10:42.147255,34.399401,-3.123011,46.229632,65.653200,97.264498,84.187556,19.930775,5.728973,50.493072,106.598368,1,1,1,False
4,2026-03-25 22:11:42.147255,30.769567,9.126396,48.885863,62.832439,70.524868,89.334008,10.467949,3.078257,135.785470,121.254020,1,1,1,False


# 3. Define função de verificação pré-decolagem

In [3]:
# Define limites mínimos e máximos para cada um dos dados
TEMP_INT_MIN_C = 18
TEMP_INT_MAX_C = 35
TEMP_EXT_MIN_C = -5
TEMP_EXT_MAX_C = 30
BAT_VOLTAGE_MIN_V = 46
BAT_VOLTAGE_MAX_V = 52
BAT_CURR_MIN_A = 20
BAT_CURR_MAX_A = 120
BAT_SOC_PERC_MIN = 60
BAT_SOC_PERC_MAX = 100
BAT_CAPACITY_MIN_AH = 80
BAT_CAPACITY_MAX_AH = 120
POWER_LOAD_MIN_KW = 5
POWER_LOAD_MAX_KW = 25
ENERGY_LOSS_MIN_PERC = 2
ENERGY_LOSS_MAX_PERC = 8
TANK_PRESSURE_MIN_BAR = 95
TANK_PRESSURE_MAX_BAR = 145

def pronto_para_decolar(telemetria: dict) -> bool:
    """
    Verifica se a aeronave está pronta para decolar baseado em sua telemetria pré-decolagem.
    
    Args:
        telemetria (dict): Dados de telemetria da aeronave.
    
    Returns:
        pronto_para_decolar (bool): Booleano indicando se a aeronave está pronta para decolar.
    """
    if telemetria["telemetry_link_status"] == 0:
        return False

    if telemetria["structural_integrity"] == 0:
        return False

    if telemetria["critical_modules_status"] == 0:
        return False

    temp_int_c = telemetria["internal_temp_c"]
    if temp_int_c < TEMP_INT_MIN_C or temp_int_c > TEMP_INT_MAX_C:
        return False

    temp_ext_c = telemetria["external_temp_c"]
    if temp_ext_c < TEMP_EXT_MIN_C or temp_ext_c > TEMP_EXT_MAX_C:
        return False

    bat_voltage_v = telemetria["battery_voltage_v"]
    if bat_voltage_v < BAT_VOLTAGE_MIN_V or bat_voltage_v > BAT_VOLTAGE_MAX_V:
        return False
    
    bat_curr_a = telemetria["battery_current_a"]
    if bat_curr_a < BAT_CURR_MIN_A or bat_curr_a > BAT_CURR_MAX_A:
        return False
    
    bat_soc_perc = telemetria["battery_soc_percentage"]
    if bat_soc_perc < BAT_SOC_PERC_MIN or bat_soc_perc > BAT_SOC_PERC_MAX:
        return False
    
    bat_capacity_ah = telemetria["battery_capacity_ah"]
    if bat_capacity_ah < BAT_CAPACITY_MIN_AH or bat_capacity_ah > BAT_CAPACITY_MAX_AH:
        return False

    power_load_kw = telemetria["power_load_kw"]
    if power_load_kw < POWER_LOAD_MIN_KW or power_load_kw > POWER_LOAD_MAX_KW:
        return False

    tank_pressure_bar = telemetria["tank_pressure_bar"]
    if tank_pressure_bar < TANK_PRESSURE_MIN_BAR or tank_pressure_bar > TANK_PRESSURE_MAX_BAR:
        return False
     
    return True

## 4. Verificação Pre-decolagem
Realiza a verificação de segurança dos dados simulados utilizando a função `pronto_para_decolar`

In [4]:
df_result = df.copy()

# Itera por cada linha do dataframe
for idx, row in df.iterrows():
    df_result.loc[idx, "is_ready_to_launch"] = pronto_para_decolar(row.to_dict())

df_result.to_csv("telemetria_resultado.csv", index=False)
df_result.head()

,timestamp,internal_temp_c,external_temp_c,battery_voltage_v,battery_current_a,battery_soc_percentage,battery_capacity_ah,power_load_kw,energy_loss_percent,estimated_takeoff_consumption_kw,tank_pressure_bar,structural_integrity,critical_modules_status,telemetry_link_status,is_anomaly,is_ready_to_launch
0,2026-03-25 22:07:42.147255,34.473606,25.185036,46.386056,51.036292,81.440589,119.163745,7.285487,2.854403,136.894277,131.103790,1,1,1,False,True
1,2026-03-25 22:08:42.147255,25.177850,-4.188213,48.465421,114.629270,75.429676,103.886668,8.477682,2.392274,125.862344,135.912748,1,1,1,False,True
2,2026-03-25 22:09:42.147255,22.049348,25.942289,48.137919,29.517034,85.197610,114.722101,12.671776,2.339611,73.772484,112.410687,1,1,1,False,True
3,2026-03-25 22:10:42.147255,34.399401,-3.123011,46.229632,65.653200,97.264498,84.187556,19.930775,5.728973,50.493072,106.598368,1,1,1,False,True
4,2026-03-25 22:11:42.147255,30.769567,9.126396,48.885863,62.832439,70.524868,89.334008,10.467949,3.078257,135.785470,121.254020,1,1,1,False,True


# 5. Teste do algoritmo de verificação de pré-decolagem
Visto que os dados com anomalia estão identificados com a flag `is_anomaly`, podemos utilizá-la para assegurar que todos os casos de anomalia foram capturados pelo algorítmo de verificação.

A verificação é feita da seguinte forma:
1. Filtra-se o dataframe de resultados por linhas onde `is_anomaly` é verdadeiro e `is_ready_to_launch` é verdadeiro.
2. Verifica-se se o dataframe filtrado está vazio indicando que não há possibilidade de lançamento em casos de anomalia.
3. Realiza a verificação oposta filtrando por linhas onde `is_anomaly` é falso e `is_ready_to_launch` é falso.
4. Verifica se o dataframe resultante está vazio indicando que não há possibilidade de abortar um lançamento quando não há anomalias.

In [5]:
# Filtra o dataframe por linhas com anomalia e prontas para decolagem
df_anomalies = df_result[(df_result["is_anomaly"] == True) & (df_result["is_ready_to_launch"] == True)]

# Assegura que o número de decolagens permitidas em dados com anomalia é igual a 0
assert(len(df_anomalies) == 0)

# Filtra o dataframe por linhas sem anomalia e com decolagem abortada
df_no_anomalies = df_result[(df_result["is_anomaly"] == False) & (df_result["is_ready_to_launch"] == False)]

# Assegura que o número de decolagens abortadas em dados sem anomalia é igual a 0
assert(len(df_no_anomalies) == 0)


# 6. Apresentação dos resultados
Exibe alguns exemplos de verificações pré-decolagens

In [6]:
# Seleciona aleatóriamente 5 linhas com anomalia e 5 linhas sem anomalia
df_anomalies = df[df["is_anomaly"] == True]
df_non_anomalies = df[df["is_anomaly"] == False]
anomaly_samples = df_anomalies.sample(5)
non_anomaly_samples = df_non_anomalies.sample(5)

# Agrega os dois dataframes
samples = pd.concat([anomaly_samples, non_anomaly_samples])

# Verifica a possibilidade de decolagem e imprime o resultado
for _, sample in samples.iterrows():
    if(pronto_para_decolar(sample.to_dict())):
        print(f"{sample.timestamp} - PRONTO PARA DECOLAR")
    else:
        print(f"{sample.timestamp} - DECOLAGEM ABORTADA")

2026-03-26 04:09:42.147255 - DECOLAGEM ABORTADA
2026-03-25 23:42:42.147255 - DECOLAGEM ABORTADA
2026-03-25 22:37:42.147255 - DECOLAGEM ABORTADA
2026-03-26 04:28:42.147255 - DECOLAGEM ABORTADA
2026-03-25 23:17:42.147255 - DECOLAGEM ABORTADA
2026-03-26 02:48:42.147255 - PRONTO PARA DECOLAR
2026-03-25 22:12:42.147255 - PRONTO PARA DECOLAR
2026-03-26 02:16:42.147255 - PRONTO PARA DECOLAR
2026-03-26 01:52:42.147255 - PRONTO PARA DECOLAR
2026-03-25 23:10:42.147255 - PRONTO PARA DECOLAR


# 7. Cálculos energéticos

Utilizando os dados simulados podemos calcular dados energéticos para verificação, nesta análise iremos calcular os seguintes dados:

### Carga total da bateria
Para simplificar o cálculo foi considerado uma bateria de uma célula similar a uma pilha. O cálculo da carga de uma bateria é dado pelo produto da tensão da bateria (V) e capacidade da bateria (Ah) o que nos dará o valor em Wh.

$C_{Total} = Tensão * Capacidade$

### Carga atual da bateria
Para calcular a carga atual basta acrescentarmos a porcentagem da bateria no cálculo de carga total.  
Assim temos a seguinte fórmula:

$C_{Atual} = Tensão * Capacidade * (Porcentagem \div 100)$

### Energia útil
Agora que temos a carga atual da bateria podemos calcular a energia útil.
De forma simplificada podemos considerar a energia útil como o produto da carga atual da bateria e a sua porcentagem de eficiência.

$E_{Util} = C_{Atual} * Eficiência$

### Autonomia inicial
Por fim podemos calcular a autonomia inicial da nave que será dada pela razão entre a energia útil e o consumo.

$Autonomia = \frac{E_{Util}}{Consumo}$

In [7]:
# Adiciona coluna para carga total
# O resultado é dividido por 1000 para converter o valor de Wh para kWh
df["battery_energy_kwh"] = (df["battery_voltage_v"] * df["battery_capacity_ah"]) / 1000

# Adiciona columna para carga atual
# O resultado é dividido por 1000 para converter o valor de Wh para kWh
df["energy_available_kwh"] = (df["battery_voltage_v"] * df["battery_capacity_ah"] * (df["battery_soc_percentage"]/100)) / 1000

# Adiciona coluna para energia útil
df["usable_energy_kwh"] = df["energy_available_kwh"] * (1 - df["energy_loss_percent"] / 100)

# Adiciona coluna para autonomia inicial
df["energetic_autonomy_h"] = df["usable_energy_kwh"] / df["estimated_takeoff_consumption_kw"]
